# 迭代器與生成器

## 學習目標

- 理解迭代器協定（`__iter__`、`__next__`）
- 會寫自訂迭代器
- 掌握生成器函式與 `yield` 的運作方式
- 了解生成器表達式的記憶體優勢
- 認識 `yield from` 與實用的 itertools 工具

---

## 1. 迭代器協定

Python 的 `for` 迴圈背後依賴兩個方法：

| 方法 | 功能 |
|------|------|
| `__iter__()` | 回傳迭代器物件本身 |
| `__next__()` | 回傳下一個值，沒有值時拋出 `StopIteration` |

In [ ]:
nums = [1, 2, 3]
it = iter(nums)       # 呼叫 nums.__iter__()
print(next(it))       # 1  ← 呼叫 it.__next__()
print(next(it))       # 2
print(next(it))       # 3
# print(next(it))     # StopIteration!

`for x in nums` 就是自動幫你做 `iter()` + 反覆 `next()` 直到 `StopIteration`。

---

## 2. 自訂迭代器

### 倒數計時器

In [ ]:
class CountDown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value

for n in CountDown(5):
    print(n, end=" ")
# 5 4 3 2 1

注意：這個迭代器是「一次性」的，用完就沒了。如果需要重複使用，`__iter__` 應該回傳新的迭代器物件。

---

## 3. 生成器函式

用 `yield` 取代 `return`，Python 自動幫你實作迭代器協定：

In [ ]:
def countdown(start):
    while start > 0:
        yield start
        start -= 1

for n in countdown(5):
    print(n, end=" ")
# 5 4 3 2 1

生成器的魔法：每次 `yield` 時函式「暫停」，下次 `next()` 時從暫停處繼續。不需要手動管理 `__iter__` 和 `__next__`。

### 與一般函式的差異

In [ ]:
def normal_func():
    return [1, 2, 3]    # 一次算完，全部放記憶體

def generator_func():
    yield 1              # 要一個，算一個
    yield 2
    yield 3

---

## 4. 生成器表達式 vs 串列推導式

In [ ]:
# 串列推導式：立刻建立完整清單，佔記憶體
squares_list = [x**2 for x in range(1_000_000)]

# 生成器表達式：只記住「怎麼算」，幾乎不佔記憶體
squares_gen = (x**2 for x in range(1_000_000))

In [ ]:
import sys

data_list = [x for x in range(100_000)]
data_gen  = (x for x in range(100_000))

print(sys.getsizeof(data_list))  # ~800,000 bytes
print(sys.getsizeof(data_gen))   #      ~200 bytes

> 原則：只需要逐一處理時用生成器表達式 `()`；需要索引、切片、重複存取時用串列 `[]`。

---

## 5. yield from

`yield from` 讓你把一個可迭代物件的元素「委派」出去，簡化巢狀生成器：

In [ ]:
def flatten(nested):
    for sublist in nested:
        yield from sublist  # 等同於 for item in sublist: yield item

data = [[1, 2], [3, 4], [5]]
print(list(flatten(data)))  # [1, 2, 3, 4, 5]

---

## 6. 實戰：資料管線

生成器最強的應用是串接成管線，每一步都不佔額外記憶體：

In [ ]:
def read_lines(filename):
    """逐行讀取檔案"""
    with open(filename) as f:
        for line in f:
            yield line.strip()

def filter_comments(lines):
    """過濾註解行"""
    for line in lines:
        if not line.startswith("#"):
            yield line

def parse_csv(lines):
    """解析 CSV 欄位"""
    for line in lines:
        yield line.split(",")

# 串接管線：讀取 → 過濾 → 解析
# pipeline = parse_csv(filter_comments(read_lines("data.csv")))
#
# for row in pipeline:
#     print(row)

整個過程一次只處理一行，即使檔案有 10 GB 也不會爆記憶體。

---

## 7. itertools 精選

標準庫 `itertools` 提供高效能的迭代工具，以下是最常用的三個：

In [ ]:
import itertools

# chain：串接多個可迭代物件
combined = itertools.chain([1, 2], [3, 4], [5])
print(list(combined))  # [1, 2, 3, 4, 5]

# islice：對迭代器做切片（不需要轉成清單）
first_three = itertools.islice(range(100), 3)
print(list(first_three))  # [0, 1, 2]

# product：笛卡兒積（取代巢狀 for 迴圈）
pairs = itertools.product("AB", [1, 2])
print(list(pairs))  # [('A',1), ('A',2), ('B',1), ('B',2)]

| 函式 | 用途 | 等同於 |
|------|------|--------|
| `chain(a, b)` | 串接多個序列 | `for x in a: yield x; for x in b: yield x` |
| `islice(it, n)` | 取前 n 個元素 | 迭代器版的 `list[:n]` |
| `product(a, b)` | 排列組合 | 巢狀 `for` 迴圈 |

---

## 總結

| 概念 | 重點 |
|------|------|
| 迭代器協定 | `__iter__` + `__next__` + `StopIteration` |
| 自訂迭代器 | 適合需要複雜狀態管理的場景 |
| 生成器函式 | `yield` 暫停/恢復，自動實作迭代器協定 |
| 生成器表達式 | `()` 取代 `[]`，幾乎不佔記憶體 |
| `yield from` | 委派子生成器，簡化巢狀結構 |
| 資料管線 | 串接生成器，逐一處理大量資料 |
| itertools | `chain`、`islice`、`product` 最常用 |

---

上一篇：[裝飾器模式](01_decorator_pattern.ipynb)
下一篇：[常見設計模式](03_design_patterns_overview.ipynb)